# LongFlow P1 -- score the 20K-step eval on GPU (not your Mac's CPU)

Runtime: **any GPU** (T4/L4/A100 all fine -- this is scoring, not
training, no VibeVoice model needed at all). Should take a few minutes,
not hours.

Drag in **`train20k_v2_eval (1).zip`** (already on your Mac from the
training run) when the cold start asks. This notebook does NOT load
VibeVoice -- it just runs Whisper-large-v3 + ECAPA on the already-
generated audio, on GPU instead of your Mac's CPU. The Mac-side script
(`score_train20k_v2.py`) does the identical computation via the same
`src/eval/metrics.py` functions -- just point `device="cuda"` instead of
the CPU default, which is the whole fix.

In [ ]:
# ===== COLD START (idempotent) -- run me first, wait for READY =====
NOTEBOOK_VERSION = "Score train20k v2 (GPU) v1.0 (2026-08-17)"
print(f"*** {NOTEBOOK_VERSION} ***")
!pip install -q faster-whisper jiwer whisper-normalizer speechbrain praat-parselmouth

import torch
assert torch.cuda.is_available(), "no GPU -- pick a GPU runtime (Runtime > Change runtime type)"

import glob, json, os, sys, zipfile

if os.path.exists("/content/LongFlow/src"):
    !cd /content/LongFlow && git pull -q
else:
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
assert os.path.exists("/content/LongFlow/src"), "clone failed -- check repo access"
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1

from src.eval.metrics import clip_metrics

assert os.path.exists("/content/train20k_v2_eval (1).zip") or os.path.exists("/content/train20k_v2_eval.zip"), \
    "DRAG the downloaded eval zip in (either filename), then re-run this cell"
zip_path = "/content/train20k_v2_eval (1).zip" if os.path.exists("/content/train20k_v2_eval (1).zip") \
    else "/content/train20k_v2_eval.zip"
os.makedirs("/content/eval_audio", exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content/eval_audio")
print(f"extracted {len(os.listdir('/content/eval_audio'))} files from {zip_path}")
print("READY")

In [ ]:
# ===== Score every checkpoint on GPU =====
AUD = "/content/eval_audio"
with open(f"{AUD}/manifest.json") as f:
    manifest = json.load(f)

STEPS = sorted(int(s) for s in manifest["checkpoints"])
results = {"held_out_per_bin": manifest["held_out_per_bin"], "steps": {}}

for step in STEPS:
    entries = manifest["checkpoints"][str(step)]
    print(f"\n=== step {step} ({len(entries)} utterances) ===")
    rows = []
    for e in entries:
        m = clip_metrics(
            f"{AUD}/{e['audio']}", e["text"], f"{AUD}/{e['teacher_audio']}", device="cuda"
        )
        rows.append({**m, "utt_id": e["utt_id"], "target_words": e["target_words"]})
        print(
            f"  {e['utt_id']} (w={e['target_words']:>4}): "
            f"wer={m['wer']:.3f}  sim={m['speaker_sim']:.3f}  "
            f"voiced={m['voiced_fraction']:.2f}  dur={m['duration_s']:.0f}s"
        )

    by_bin = {}
    for r in rows:
        by_bin.setdefault(r["target_words"], []).append(r)
    bin_summary = {
        tw: {
            "n": len(rs),
            "wer_median": sorted(r["wer"] for r in rs)[len(rs) // 2],
            "sim_median": sorted(r["speaker_sim"] for r in rs)[len(rs) // 2],
        }
        for tw, rs in sorted(by_bin.items())
    }
    overall = {
        "n": len(rows),
        "wer_median": sorted(r["wer"] for r in rows)[len(rows) // 2],
        "sim_median": sorted(r["speaker_sim"] for r in rows)[len(rows) // 2],
    }
    print(f"  overall: wer_median={overall['wer_median']:.3f}  sim_median={overall['sim_median']:.3f}")
    results["steps"][step] = {"overall": overall, "by_bin": bin_summary, "rows": rows}

print("\n" + "=" * 60)
print("SUMMARY (overall median WER / speaker-sim per checkpoint)")
print("=" * 60)
for step in STEPS:
    o = results["steps"][step]["overall"]
    print(f"  step {step:>6}: n={o['n']:>3}  wer={o['wer_median']:.3f}  sim={o['sim_median']:.3f}")

if len(STEPS) >= 2:
    best_step = min(STEPS, key=lambda s: results["steps"][s]["overall"]["wer_median"])
    last_step = STEPS[-1]
    if best_step != last_step:
        print(
            f"\nWATCH: best held-out WER is at step {best_step}, not the final "
            f"step {last_step} -- possible overfitting past {best_step}, matching "
            "the July E3 signature (20K optimal / 80K overfit). Consider "
            f"{best_step} as the operating checkpoint instead of {last_step}."
        )
    else:
        print(f"\nFinal step {last_step} has the best held-out WER -- no overfitting signature.")

with open("/content/train20k_v2_metrics.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nwrote /content/train20k_v2_metrics.json")

In [ ]:
# ===== Download the (tiny) results JSON =====
from google.colab import files as colab_files
colab_files.download("/content/train20k_v2_metrics.json")